# AdamW Optimizer: A Deep Dive

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/transformer-tutorial/blob/main/adamw_tutorial.ipynb)

---

本 notebook 系统讲解 AdamW 优化器的核心原理，包括：

1. **梯度下降**：参数如何利用导数更新
2. **Adam 优化器**：动量 + 自适应学习率
3. **L2 正则化**：什么是 Weight Decay，为什么需要它
4. **AdamW**：Adam 与 Decoupled Weight Decay 的结合
5. **PyTorch 实战**：在神经网络中使用 AdamW，Adam vs AdamW 对比


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from copy import deepcopy

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 110

---
## 1. 梯度下降：参数是如何用导数更新的

训练神经网络的目标是最小化损失函数 $\mathcal{L}(\theta)$。

**核心思想**：沿着梯度（导数）的反方向移动参数，损失就会下降。

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_{\theta} \mathcal{L}$$

- $\theta$：模型参数（权重）
- $\eta$：学习率（步长大小）
- $\nabla_{\theta} \mathcal{L}$：损失对参数的偏导数（梯度），由 **反向传播（Backpropagation）** 自动计算

**为什么是梯度的反方向？** 梯度指向函数增长最快的方向，反方向则是下降最快的方向。

In [ ]:
# ---- 可视化：梯度下降在一个简单抛物线上的过程 ----
def loss_fn(x):
    return (x - 3.0) ** 2  # 最小值在 x=3

def grad_fn(x):
    return 2 * (x - 3.0)   # 解析梯度

lr = 0.15
x = torch.tensor(-2.0)
history = [x.item()]

for _ in range(12):
    g = grad_fn(x)
    x = x - lr * g          # 梯度下降更新
    history.append(x.item())

xs = np.linspace(-3, 6, 300)
ys = (xs - 3.0) ** 2

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs, ys, 'steelblue', lw=2, label='$\\mathcal{L}(x) = (x-3)^2$')
for i, xi in enumerate(history):
    color = plt.cm.Reds(i / len(history))
    ax.scatter(xi, loss_fn(torch.tensor(xi)).item(), color=color, zorder=5, s=60)
    if i < len(history) - 1:
        ax.annotate('', xy=(history[i+1], loss_fn(torch.tensor(history[i+1])).item()),
                    xytext=(xi, loss_fn(torch.tensor(xi)).item()),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))

ax.axvline(3, color='green', linestyle='--', alpha=0.6, label='Global minimum x=3')
ax.set_xlabel('Parameter x')
ax.set_ylabel('Loss')
ax.set_title('Gradient Descent: Step by Step')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Starting x: -2.0  →  Final x: {history[-1]:.4f}  (target: 3.0)')

---
## 2. Adam 优化器

普通梯度下降有两个问题：
1. 所有参数用同一学习率，但不同参数的梯度量级差异很大
2. 梯度方向抖动大，收敛慢

**Adam（Adaptive Moment Estimation）** 解决了这两个问题，核心是维护两个滑动平均：

### 算法步骤（每步 $t$）

| 变量 | 含义 | 公式 |
|------|------|------|
| $g_t$ | 当前梯度 | $\nabla_{\theta}\mathcal{L}$ |
| $m_t$ | 一阶矩（梯度的指数平均，**动量**） | $m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$ |
| $v_t$ | 二阶矩（梯度平方的指数平均，**自适应缩放**） | $v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$ |
| $\hat{m}_t$ | 偏差修正后的一阶矩 | $\hat{m}_t = m_t / (1 - \beta_1^t)$ |
| $\hat{v}_t$ | 偏差修正后的二阶矩 | $\hat{v}_t = v_t / (1 - \beta_2^t)$ |
| $\theta_t$ | 参数更新 | $\theta_t = \theta_{t-1} - \eta \cdot \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$ |

**默认超参数**：$\beta_1 = 0.9$，$\beta_2 = 0.999$，$\epsilon = 10^{-8}$

- $\beta_1 = 0.9$ 意味着动量主要来自过去 10 步的梯度
- $\sqrt{\hat{v}_t}$ 作为分母，梯度大的参数学习率自动变小，梯度小的参数学习率自动变大

In [ ]:
# ---- 从零手写 Adam，理解每一步 ----
class AdamFromScratch:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.t = 0  # 步数
        # 为每个参数初始化一阶矩和二阶矩，初始值为 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g = p.grad

            # Step 1: 更新一阶矩（动量）
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g

            # Step 2: 更新二阶矩（自适应缩放）
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2

            # Step 3: 偏差修正（早期步数时一阶/二阶矩被初始化为0，导致低估）
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            # Step 4: 更新参数
            p.data -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


# 验证：用手写 Adam 优化一个简单二次函数
w = torch.tensor([5.0], requires_grad=True)
optimizer = AdamFromScratch([w], lr=0.1)
losses = []

for step in range(60):
    optimizer.zero_grad()
    loss = (w - 2.5) ** 2   # 目标：w → 2.5
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses, color='steelblue')
axes[0].set_title('Adam: Loss Curve')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')

# 可视化偏差修正的效果
t_vals = np.arange(1, 51)
beta1, beta2 = 0.9, 0.999
bias1 = 1 - beta1 ** t_vals
bias2 = 1 - beta2 ** t_vals
axes[1].plot(t_vals, bias1, label='$1 - \\beta_1^t$ (1st moment correction)')
axes[1].plot(t_vals, bias2, label='$1 - \\beta_2^t$ (2nd moment correction)', linestyle='--')
axes[1].axhline(1, color='gray', linestyle=':', alpha=0.7)
axes[1].set_title('Bias Correction Factor over Steps')
axes[1].set_xlabel('Step t')
axes[1].set_ylabel('Correction Factor')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'Final w = {w.item():.4f}  (target: 2.5)')

---
## 3. L2 正则化（Weight Decay）是什么？

### 过拟合问题
当模型参数过大时，模型会过度拟合训练数据，泛化能力差。正则化通过惩罚大权重来约束模型复杂度。

### L2 正则化
在损失函数中加入参数的 L2 范数惩罚项：

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{task}} + \frac{\lambda}{2} \|\theta\|^2$$

对参数求导：

$$\nabla_{\theta} \mathcal{L}_{\text{total}} = \nabla_{\theta} \mathcal{L}_{\text{task}} + \lambda \theta$$

代入梯度下降更新规则：

$$\theta_{t+1} = \theta_t - \eta (g_t + \lambda \theta_t) = \underbrace{(1 - \eta\lambda)}_{\text{weight decay}} \theta_t - \eta g_t$$

**效果**：每步更新前先将参数乘以 $(1 - \eta\lambda)$，即把参数向 0 拉近，防止参数爆炸。

### Adam 中 L2 正则化的问题

在 Adam 中，L2 正则项 $\lambda\theta$ 会被加进梯度 $g_t$，然后被自适应缩放因子 $\sqrt{\hat{v}_t}$ 除掉：

$$\theta_t = \theta_{t-1} - \eta \cdot \frac{\hat{m}_t + \lambda\theta}{\sqrt{\hat{v}_t} + \epsilon}$$

这意味着**权重衰减的强度被自适应因子调制了**，对于梯度历史大的参数，L2 惩罚被自动缩小，导致正则效果不一致 — 这正是 AdamW 要解决的问题。

In [ ]:
# ---- 可视化：L2 正则化如何约束权重大小 ----
steps = 80
lr_demo = 0.05

def train_no_reg(steps):
    w = torch.tensor([8.0], requires_grad=True)
    ws = [w.item()]
    for _ in range(steps):
        loss = 0.5 * w ** 2 + 5.0  # 最小值在 w=0，但加了常数偏移
        loss.backward()
        with torch.no_grad():
            w -= lr_demo * w.grad
        w.grad.zero_()
        ws.append(w.item())
    return ws

def train_with_l2(steps, lam):
    w = torch.tensor([8.0], requires_grad=True)
    ws = [w.item()]
    for _ in range(steps):
        task_loss = torch.tensor(5.0)   # 假设任务损失梯度为 0（w 对任务无影响）
        l2_loss = (lam / 2) * w ** 2   # L2 惩罚
        loss = task_loss + l2_loss
        loss.backward()
        with torch.no_grad():
            w -= lr_demo * w.grad
        w.grad.zero_()
        ws.append(w.item())
    return ws

ws_no_reg = train_no_reg(steps)
ws_l2_small = train_with_l2(steps, lam=0.1)
ws_l2_large = train_with_l2(steps, lam=0.5)

plt.figure(figsize=(9, 4))
plt.plot(ws_no_reg, label='No regularization', color='gray', linestyle='--')
plt.plot(ws_l2_small, label='L2 λ=0.1', color='orange')
plt.plot(ws_l2_large, label='L2 λ=0.5', color='red')
plt.axhline(0, color='green', linestyle=':', alpha=0.7, label='w = 0 (L2 target)')
plt.xlabel('Step')
plt.ylabel('Weight value')
plt.title('L2 Regularization: Weight Decay Effect')
plt.legend()
plt.tight_layout()
plt.show()
print('L2 正则化将权重向 0 拉近，λ 越大，衰减越快')

---
## 4. AdamW：解耦权重衰减（Decoupled Weight Decay）

**论文**：[Decoupled Weight Decay Regularization (Loshchilov & Hutter, 2019)](https://arxiv.org/abs/1711.05101)

### 关键改进

AdamW 将权重衰减**直接作用于参数**，而不是加进梯度：

| | Adam + L2 | AdamW |
|--|-----------|-------|
| 更新公式 | $\theta_t = \theta_{t-1} - \eta \cdot \frac{\hat{m}(g + \lambda\theta)}{\sqrt{\hat{v}(g + \lambda\theta)} + \epsilon}$ | $\theta_t = (1 - \eta\lambda)\theta_{t-1} - \eta \cdot \frac{\hat{m}(g)}{\sqrt{\hat{v}(g)} + \epsilon}$ |
| 权重衰减强度 | 被自适应因子调制，不均匀 | 固定，对所有参数一致 |
| 正则效果 | 弱（实际上是 L2 正则的近似） | 强（真正的权重衰减） |

### AdamW 完整算法

```
for t = 1, 2, ... do:
    g_t = ∇L(θ_{t-1})                         # 计算梯度
    m_t = β₁·m_{t-1} + (1-β₁)·g_t             # 一阶矩
    v_t = β₂·v_{t-1} + (1-β₂)·g_t²            # 二阶矩
    m̂_t = m_t / (1-β₁ᵗ)                       # 偏差修正
    v̂_t = v_t / (1-β₂ᵗ)                       # 偏差修正
    θ_t = (1 - η·λ)·θ_{t-1}                   # ← 先做权重衰减 (AdamW 关键)
          - η · m̂_t / (√v̂_t + ε)             # ← 再做 Adam 更新
```

In [ ]:
# ---- 从零手写 AdamW，与 Adam 对比 ----
class AdamWFromScratch:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.wd = weight_decay
        self.t = 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g = p.grad  # 注意：这里的梯度不含 L2 项

            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            # AdamW 关键：先做权重衰减（直接作用于参数，与梯度无关）
            p.data *= (1 - self.lr * self.wd)
            # 再做 Adam 梯度更新
            p.data -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


# 对比：在一个大初始权重上，Adam vs AdamW 的权重衰减行为
def run_optimizer(OptimizerClass, init_w=10.0, steps=100, **kwargs):
    w = torch.tensor([init_w], requires_grad=True)
    opt = OptimizerClass([w], **kwargs)
    ws = [w.item()]
    for _ in range(steps):
        opt.zero_grad()
        # 任务损失的梯度很小（模拟参数已接近最优但仍然很大的情形）
        loss = 0.01 * w
        loss.backward()
        opt.step()
        ws.append(w.item())
    return ws

ws_adam   = run_optimizer(AdamFromScratch,   lr=0.01, weight_decay=0) if False else None

# 使用 PyTorch 内置优化器对比
def run_torch_optimizer(optimizer_name, init_w=10.0, steps=100, **kwargs):
    w = torch.tensor([init_w], requires_grad=True)
    if optimizer_name == 'adam':
        opt = optim.Adam([w], **kwargs)
    else:
        opt = optim.AdamW([w], **kwargs)
    ws = [w.item()]
    for _ in range(steps):
        opt.zero_grad()
        loss = 0.01 * w
        loss.backward()
        opt.step()
        ws.append(w.item())
    return ws

steps = 120
ws_adam_no_wd  = run_torch_optimizer('adam',  lr=0.05, weight_decay=0.0,  steps=steps)
ws_adam_l2     = run_torch_optimizer('adam',  lr=0.05, weight_decay=0.1,  steps=steps)
ws_adamw       = run_torch_optimizer('adamw', lr=0.05, weight_decay=0.1,  steps=steps)

plt.figure(figsize=(9, 4))
plt.plot(ws_adam_no_wd, label='Adam (no decay)', linestyle='--', color='gray')
plt.plot(ws_adam_l2,    label='Adam + L2 (weight_decay=0.1)', color='orange')
plt.plot(ws_adamw,      label='AdamW (weight_decay=0.1)', color='steelblue', lw=2)
plt.axhline(0, color='green', linestyle=':', alpha=0.6)
plt.xlabel('Step')
plt.ylabel('Weight value')
plt.title('Adam vs AdamW: Weight Decay Comparison')
plt.legend()
plt.tight_layout()
plt.show()
print('AdamW 的权重衰减更强且更稳定，不受自适应因子的干扰')

---
## 5. 在 PyTorch 神经网络中使用 AdamW

### 5.1 构建网络与数据集

In [ ]:
# ---- 生成二分类数据集 ----
from torch.utils.data import DataLoader, TensorDataset

n = 400
X_pos = torch.randn(n // 2, 2) + torch.tensor([1.5, 1.5])
X_neg = torch.randn(n // 2, 2) + torch.tensor([-1.5, -1.5])
X = torch.cat([X_pos, X_neg])
y = torch.cat([torch.ones(n // 2), torch.zeros(n // 2)]).long()

# 划分训练/测试集
idx = torch.randperm(n)
X_train, y_train = X[idx[:300]], y[idx[:300]]
X_test,  y_test  = X[idx[300:]], y[idx[300:]]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

# ---- 定义神经网络 ----
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)

print('Dataset: 400 samples, 2 classes, 2 features')
print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')
print()
print('Model architecture:')
print(MLP())

### 5.2 配置 AdamW 的三种常见模式

PyTorch 中使用 `torch.optim.AdamW`：

```python
# 模式 1：全局统一的 weight_decay
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

# 模式 2：不对 bias 和 LayerNorm 做权重衰减（推荐用法）
decay_params    = [p for n, p in model.named_parameters() if 'bias' not in n]
no_decay_params = [p for n, p in model.named_parameters() if 'bias' in n]
optimizer = torch.optim.AdamW([
    {'params': decay_params,    'weight_decay': 1e-2},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=1e-3)

# 模式 3：配合学习率调度器（常见于 Transformer 训练）
from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(optimizer, T_max=100)
```

In [ ]:
# ---- 训练函数 ----
def train(model, optimizer, epochs=60):
    criterion = nn.CrossEntropyLoss()
    train_losses, test_accs = [], []
    weight_norms = []  # 追踪权重范数，观察正则化效果

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()      # 1. 清空梯度
            logits = model(xb)         # 2. 前向传播
            loss = criterion(logits, yb)  # 3. 计算损失
            loss.backward()            # 4. 反向传播（自动计算所有参数的梯度）
            optimizer.step()           # 5. 用梯度更新参数（AdamW 在这里执行）
            epoch_loss += loss.item()

        # 测试集准确率
        model.eval()
        with torch.no_grad():
            preds = model(X_test).argmax(dim=1)
            acc = (preds == y_test).float().mean().item()

        # 计算所有参数的 L2 范数
        total_norm = sum(p.norm().item() ** 2 for p in model.parameters()) ** 0.5

        train_losses.append(epoch_loss / len(train_loader))
        test_accs.append(acc)
        weight_norms.append(total_norm)

    return train_losses, test_accs, weight_norms


# ---- 对比三种配置 ----
epochs = 80
configs = {
    'Adam (no wd)':    optim.Adam( MLP().parameters(), lr=1e-2, weight_decay=0.0),
    'Adam + L2':       optim.Adam( MLP().parameters(), lr=1e-2, weight_decay=1e-2),
    'AdamW':           optim.AdamW(MLP().parameters(), lr=1e-2, weight_decay=1e-2),
}

results = {}
torch.manual_seed(42)
for name, opt in configs.items():
    torch.manual_seed(42)
    model = MLP()
    # 重建优化器绑定新模型
    if 'AdamW' in name:
        optimizer = optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-2)
    elif 'L2' in name:
        optimizer = optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-2)
    else:
        optimizer = optim.Adam(model.parameters(), lr=1e-2, weight_decay=0.0)
    losses, accs, norms = train(model, optimizer, epochs=epochs)
    results[name] = {'losses': losses, 'accs': accs, 'norms': norms}
    print(f'{name:20s}  Final Test Acc: {accs[-1]:.3f}  Weight Norm: {norms[-1]:.2f}')

In [ ]:
# ---- 可视化训练结果 ----
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {'Adam (no wd)': 'gray', 'Adam + L2': 'orange', 'AdamW': 'steelblue'}

for name, data in results.items():
    lw = 2.5 if name == 'AdamW' else 1.5
    axes[0].plot(data['losses'], label=name, color=colors[name], lw=lw)
    axes[1].plot(data['accs'],   label=name, color=colors[name], lw=lw)
    axes[2].plot(data['norms'],  label=name, color=colors[name], lw=lw)

axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=9)

axes[1].set_title('Test Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=9)

axes[2].set_title('Weight L2 Norm (Regularization Effect)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('||θ||₂')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 6. 逐步追踪参数更新过程

下面我们用一个小例子，打印出 AdamW 每步更新的完整中间值，看清楚参数是如何被导数驱动更新的。

In [ ]:
# ---- 逐步追踪 AdamW 参数更新 ----
torch.manual_seed(0)
model = nn.Linear(2, 1, bias=False)
print(f'Initial weight: {model.weight.data}')

optimizer = optim.AdamW(model.parameters(), lr=0.1, betas=(0.9, 0.999),
                        eps=1e-8, weight_decay=0.01)

x_sample = torch.tensor([[1.0, 2.0]])
y_target = torch.tensor([[3.0]])
criterion = nn.MSELoss()

print(f'\n{"Step":<5} {"Loss":<10} {"Gradient":<30} {"Weight":<30}')
print('-' * 80)

for step in range(1, 6):
    optimizer.zero_grad()               # 清空上一步的梯度
    output = model(x_sample)            # 前向传播：y_hat = w @ x
    loss = criterion(output, y_target)  # 计算损失：MSE
    loss.backward()                     # 反向传播：计算 dL/dw

    grad_str   = str(model.weight.grad.data.numpy().round(4))
    weight_str = str(model.weight.data.numpy().round(4))
    print(f'{step:<5} {loss.item():<10.4f} {grad_str:<30} {weight_str:<30}', end='')

    optimizer.step()                    # AdamW 更新参数

    weight_after = str(model.weight.data.numpy().round(4))
    print(f'  → {weight_after}')

print('\n步骤说明：')
print('1. forward() 计算预测值')
print('2. loss.backward() 用链式法则自动求 dL/dw（即梯度列中的值）')
print('3. optimizer.step() 用梯度+动量+自适应缩放+权重衰减更新参数')

---
## 7. 总结对比

| 特性 | SGD | Adam | AdamW |
|------|-----|------|-------|
| 自适应学习率 | ❌ | ✅ | ✅ |
| 动量 | 可选 | ✅ | ✅ |
| 权重衰减（正确实现） | ✅ | ❌ | ✅ |
| 适合 Transformer | ❌ | 一般 | ✅ |
| PyTorch 类 | `optim.SGD` | `optim.Adam` | `optim.AdamW` |

### 何时选择 AdamW？

- **Transformer / BERT / GPT 类模型**：这些模型的标准配置
- **需要强正则化时**：参数量大，容易过拟合
- **NLP / CV 预训练微调**：HuggingFace 默认使用 AdamW

### 典型超参数推荐

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,           # Transformer 微调常用
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=1e-2, # 0.01 是常用默认值
)
```

### 参考文献
- [Adam: A Method for Stochastic Optimization (Kingma & Ba, 2015)](https://arxiv.org/abs/1412.6980)
- [Decoupled Weight Decay Regularization (Loshchilov & Hutter, 2019)](https://arxiv.org/abs/1711.05101)
- [PyTorch AdamW Docs](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html)